## 9.2 正则化 - 随机失活Pytorch实现

#### 1. 为什么需要单独学习 Dropout 的 PyTorch 实现
在上一小节中，我们已经学习了 Dropout 的原理：
* 训练时随机关闭一部分神经元
* 测试时使用完整网络

并且我们也知道了：
* Dropout 属于正则化方法
* 目的是减少过拟合
* 提高泛化能力

但是在真正写代码时，下面几个地方混淆：
1. Dropout 到底怎么写
2. Dropout 应该放在模型的哪里
3. 训练和测试时为什么表现不一样
4. model.train() 和 model.eval() 与 Dropout 的关系是什么

#### 2. Dropout 在 PyTorch 中的核心类

##### 2.1 最常用的类：nn.Dropout
在 PyTorch 中，最常用的 Dropout 类是：

`torch.nn.Dropout`

基本写法：
```
import torch.nn as nn
dropout = nn.Dropout(p=0.5)
```

这里的：

`p=0.5`

表示：

`每个神经元有 50% 的概率被丢弃 `

注意这里一定要记住：
* p 表示丢弃概率
* 不是保留概率

##### 2.2 Dropout 在训练时做了什么
假设某一层输出为：

`x = [2, 4, 6, 8]`

如果：

`p = 0.5`

那么训练时，PyTorch 可能会随机生成一个 mask：

`mask = [1, 0, 1, 0]`

于是输出变成：

`[2, 0, 6, 0]`

但 PyTorch 不会停在这里，而是还会自动做 缩放：

`[2, 0, 6, 0] / 0.5 = [4, 0, 12, 0]`

也就是说，PyTorch 默认使用的是：
* Inverted Dropout

训练时自动完成：
* 随机失活 + 按 1/(1-p) 缩放

#### 3. 最基础的 Dropout 使用方式

##### 3.1 单独创建一个 Dropout 层

In [1]:
import torch 
import torch.nn as nn 

dropout = nn.Dropout(0.5)

##### 3.2 对张量直接使用

In [2]:
X = torch.tensor([1.0, 2.0, 3.0, 4.0])

# 开启训练模式

dropout.train()

Y = dropout(X)

print(Y)

tensor([2., 0., 0., 0.])


解释：
* 原始输入：[1, 2, 3, 4]
* 某些元素被置 0
* 剩余元素除以 1-p = 0.5
* 所以保留下来的值乘了 2

##### 3.3 再看测试模式

In [3]:
dropout.eval()

y = dropout(X)

print(y)

tensor([1., 2., 3., 4.])


也就是说：
* 测试时 Dropout 不再随机失活

#### 4. Dropout 与 train() / eval() 的关系
这是 Dropout 在 PyTorch 中最重要的知识点之一。

##### 4.1 model.train() 的作用
当你写：

`model.train()`

表示：

`模型进入训练模式`

此时：
* Dropout 生效
* BatchNorm 使用当前 batch 的统计量

对于 Dropout 来说，这意味着：
* 神经元会被随机关闭

##### 4.2 model.eval() 的作用
当你写：

`model.eval()`

表示：

`模型进入评估 / 测试模式`

此时：
* Dropout 关闭
* BatchNorm 使用训练阶段累计的均值和方差

对于 Dropout 来说，这意味着：
* 不再随机失活
* 使用完整网络

##### 4.3 为什么必须切换模式
如果你在验证集 / 测试集阶段仍然使用：

`model.train()`

那么 Dropout 还会继续随机关闭神经元，导致：
* 预测结果不稳定
* 每次结果都可能不同

所以必须记住：
* 训练阶段：model.train()
* 验证 / 测试阶段：model.eval()

#### 5. Dropout 在模型中的标准写法

##### 5.1 最常见写法：定义为层属性

In [5]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(100,64) # 输入层到隐藏层
        self.relu = nn.ReLU() # 激活函数
        self.dropout = nn.Dropout(0.5) # Dropout层
        self.layer_2 = nn.Linear(64,10) # 隐藏层到输出层
    
    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_2(x)
        return x

##### 5.2 这一段代码的执行流程
```
输入 x
→ 全连接层 layer_1
→ 激活函数 ReLU
→ Dropout
→ 输出层 layer_2
```

##### 5.3 为什么通常放在激活函数后面
因为激活函数输出的是：
* 这一层真正传给下一层的特征

Dropout 的目的就是：
* 随机屏蔽部分特征表达

所以最常见的放置方式是：

`Linear → ReLU → Dropout`

#### 6. 完整训练流程中的 Dropout 写法

##### 6.1 定义模型结构

In [7]:
class Moodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(100, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.layer_2 = nn.Linear(64, 32)
        self.relu_2 = nn.ReLU()
        self.dropout_2 = nn.Dropout(0.3)
        self.layer_3 = nn.Linear(32, 10)
    
    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_2(x)
        x = self.relu_2(x)
        x = self.dropout_2(x)
        x = self.layer_3(x)
        return x

##### 6.2 定义损失函数 + 优化器（参数优化） + scheduler

In [14]:
model = Moodel()
# 1. 定义损失函数，对于多分类，使用cross-entropy损失函数
criterion = nn.CrossEntropyLoss()

# 2. 定义优化器，使用Adam优化器
optimizer =  torch.optim.Adam(
    model.parameters(),
    lr = 0.1
)

# 3. 定义 scheduler，使用 ReduceLROnPlateau 学习率调度器
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer = optimizer,
    mode = 'min',
    factor = 0.1,
    patience = 5
)

##### 6.3 准备模拟数据

In [10]:
X_train = torch.randn(500, 100) # 500个样本，每个样本100维特征
y_train = torch.randint(0, 10, (500,)) # 500个样本的标签，范围在0-9之间

X_val = torch.randn(100, 100) # 100个样本，每个样本100维特征
y_val = torch.randint(0, 10, (100,)) # 100个样本的标签，范围在0-9之间

# 构建Dataset 和 DataLoader
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32)


##### 6.4 定义 train 方法 + val 方法

In [12]:
def train(model, train_loader, criterion, optimizer):
    model.train() # 设置模型为训练模式
    total_loss = 0.0 # 累积损失
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad() # 清空梯度
        y_pred = model(X_batch) # 前向传播
        loss = criterion(y_pred, y_batch) # 计算损失
        loss.backward() # 反向传播
        optimizer.step() # 更新参数
        total_loss += loss.item() # 累积损失
    
    train_loss = total_loss / len(train_loader) # 计算平均损失
    return train_loss

def eval(model, val_loader, criterion):
    model.eval() # 设置模型为评估模式
    total_loss = 0.0 # 累积损失
    with torch.no_grad(): # 禁止梯度计算
        for X_val, y_val in val_loader:
            y_pred = model(X_val) # 前向传播
            loss = criterion(y_pred, y_val) # 计算损失
            total_loss += loss.item() # 累积损失
    val_loss = total_loss / len(val_loader) # 计算平均损失
    return val_loss

##### 6.5 训练主循环

In [15]:
total_epochs = 100
for epoch in range(1, total_epochs + 1):
    train_loss = train(model, train_loader, criterion, optimizer)
    val_loss = eval(model, val_loader, criterion)
    scheduler.step(val_loss) # 更新学习率
    print(f'Epoch {epoch}/{total_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

Epoch 1/100, Train Loss: 2.5437, Val Loss: 2.2941
Epoch 2/100, Train Loss: 2.3119, Val Loss: 2.2862
Epoch 3/100, Train Loss: 2.3204, Val Loss: 2.2769
Epoch 4/100, Train Loss: 2.3021, Val Loss: 2.2909
Epoch 5/100, Train Loss: 2.3257, Val Loss: 2.2814
Epoch 6/100, Train Loss: 2.3078, Val Loss: 2.2633
Epoch 7/100, Train Loss: 2.3011, Val Loss: 2.2914
Epoch 8/100, Train Loss: 2.3004, Val Loss: 2.2828
Epoch 9/100, Train Loss: 2.3261, Val Loss: 2.2773
Epoch 10/100, Train Loss: 2.3099, Val Loss: 2.2728
Epoch 11/100, Train Loss: 2.3014, Val Loss: 2.2844
Epoch 12/100, Train Loss: 2.3012, Val Loss: 2.2688
Epoch 13/100, Train Loss: 2.2945, Val Loss: 2.2716
Epoch 14/100, Train Loss: 2.2934, Val Loss: 2.2749
Epoch 15/100, Train Loss: 2.2935, Val Loss: 2.2759
Epoch 16/100, Train Loss: 2.2931, Val Loss: 2.2748
Epoch 17/100, Train Loss: 2.2924, Val Loss: 2.2783
Epoch 18/100, Train Loss: 2.2873, Val Loss: 2.2795
Epoch 19/100, Train Loss: 2.2926, Val Loss: 2.2797
Epoch 20/100, Train Loss: 2.2922, Val Lo

##### 7. Dropout 放在哪里最合适
**1️⃣ 最常见位置：全连接层之间**

对于 MLP，最常见的位置是：
* 隐藏层之间
* 尤其是全连接层之间

例如：

`fc1 → relu → dropout → fc2`

**2️⃣ 为什么输出层通常不用 Dropout**

输出层负责：
* 给出最终预测结果

如果在输出层使用 Dropout，就相当于：
* 在最终决策前随机丢信息
* 这通常不合理

所以一般规律是：
* 隐藏层可以用
* 输出层通常不用

#### 8. PyTorch 中 Dropout 的常见变体
除了最基础的：

`nn.Dropout`

PyTorch 还提供：

**1️⃣ nn.Dropout1d**

常用于某些一维特征场景。

**2️⃣ nn.Dropout2d**

常用于卷积网络的特征图。

**3️⃣ nn.Dropout3d**

常用于三维数据。

#### 9. 使用 Dropout 时的常见错误
**1️⃣ 忘记写 model.eval()**

这会导致验证 / 测试时 Dropout 还在随机生效。

结果：
* 测试结果波动很大

**2️⃣ 在输出层使用 Dropout**

通常不推荐。

**3️⃣ Dropout 比例设置过大**

会让模型训练困难。

**4️⃣ 误以为 Dropout 在测试时也会随机关闭神经元**

这是错误的。

测试时，Dropout 在 PyTorch 中会自动关闭。
